# 03 — Modélisation + MLflow

**Prérequis :** datasets `fe_v1` dans `data/processed/*.parquet` (voir `choix_techniques.txt`).

**Résultat baseline (split valid, hors CV) :** LightGBM ≈ **AUC 0.78** sur valid.

**Réalisé dans le projet (scripts `src/`) :**
- tracking MLflow (SQLite `mlflow.db`) + UI ;
- validation croisée StratifiedKFold (5 plis), multi-modèles + seuil métier ;
- Optuna sur LightGBM + seuil* sur valid ;
- SHAP global / local ;
- Model Registry `credit-scoring-lgbm@champion` + serving HTTP testé.

Avant d’exécuter les cellules MLflow : lancer l’UI (même URI SQLite que le tracking).


## 0. Setup

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import mlflow
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path("..").resolve()
PROC = ROOT / "data" / "processed"
meta = json.loads((PROC / "meta.json").read_text(encoding="utf-8"))
print("fe_version:", meta["fe_version"])
print("train/valid:", meta["train_rows"], meta["valid_rows"])

## 1. Charger processed + séparer X / y

In [ ]:
train = pd.read_parquet(PROC / "train.parquet")
valid = pd.read_parquet(PROC / "valid.parquet")

y_train = train["TARGET"].astype(int)
y_valid = valid["TARGET"].astype(int)
X_train = train.drop(columns=["TARGET", "SK_ID_CURR"])
X_valid = valid.drop(columns=["TARGET", "SK_ID_CURR"])

cat_cols = X_train.select_dtypes(include=["object", "string"]).columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]
print(f"features={X_train.shape[1]} | num={len(num_cols)} | cat={len(cat_cols)}")

## 2. Métriques métier

Protocole de coût retenu : **FN = 10**, **FP = 1**  
(`coût = 10 × faux négatifs + 1 × faux positifs`).  
Le seuil de décision est ensuite optimisé pour minimiser ce coût (pas fixé à 0.5 par défaut).


In [ ]:
FN_COST = 10
FP_COST = 1


def business_cost(y_true, y_pred, fn_cost=FN_COST, fp_cost=FP_COST):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    return fn_cost * fn + fp_cost * fp, fn, fp


def eval_at_threshold(y_true, proba, threshold=0.5):
    pred = (proba >= threshold).astype(int)
    cost, fn, fp = business_cost(y_true, pred)
    return {
        "threshold": threshold,
        "auc": float(roc_auc_score(y_true, proba)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1": float(f1_score(y_true, pred)),
        "recall": float(recall_score(y_true, pred)),
        "business_cost": float(cost),
        "fn": fn,
        "fp": fp,
    }

## 3. Brancher MLflow

Lancer l’UI depuis la racine du projet (backend SQLite) :

```powershell
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db --port 5000
```

Ouvrir http://127.0.0.1:5000  

Le tracking du notebook doit pointer vers **le même** URI SQLite.


In [ ]:
# === JALON : activer après validation FE ===
# Même URI que l'UI (sqlite), sinon les runs n'apparaissent pas dans mlflow ui
MLFLOW_TRACKING_URI = f"sqlite:///{(ROOT / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "credit-scoring-fe_v1"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)
print("tracking_uri =", MLFLOW_TRACKING_URI)
print("experiment   =", EXPERIMENT_NAME)


## 4. Baseline Logistic Regression (pipeline sklearn)

In [ ]:
preprocess_lr = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("ohe", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ]
)

pipe_lr = Pipeline(
    steps=[
        ("preprocess", preprocess_lr),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [ ]:
with mlflow.start_run(run_name="logreg_baseline_fe_v1"):
    mlflow.set_tags({"fe_version": meta["fe_version"], "model_family": "logreg"})
    mlflow.log_params({"class_weight": "balanced", "max_iter": 1000, "threshold": 0.5})

    pipe_lr.fit(X_train, y_train)
    proba = pipe_lr.predict_proba(X_valid)[:, 1]
    metrics = eval_at_threshold(y_valid, proba, threshold=0.5)
    mlflow.log_metrics({k: v for k, v in metrics.items() if k != "threshold"})
    mlflow.sklearn.log_model(pipe_lr, artifact_path="model")
    print(metrics)

## 5. LightGBM (catégorielles natives)

In [ ]:
X_train_lgb = X_train.copy()
X_valid_lgb = X_valid.copy()
for c in cat_cols:
    X_train_lgb[c] = X_train_lgb[c].astype("category")
    X_valid_lgb[c] = X_valid_lgb[c].astype("category")

lgb = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

In [ ]:
with mlflow.start_run(run_name="lightgbm_baseline_fe_v1"):
    mlflow.set_tags({"fe_version": meta["fe_version"], "model_family": "lightgbm"})
    mlflow.log_params(
        {
            "n_estimators": 300,
            "learning_rate": 0.05,
            "num_leaves": 31,
            "class_weight": "balanced",
            "threshold": 0.5,
        }
    )
    lgb.fit(X_train_lgb, y_train, categorical_feature=cat_cols)
    proba = lgb.predict_proba(X_valid_lgb)[:, 1]
    metrics = eval_at_threshold(y_valid, proba, threshold=0.5)
    mlflow.log_metrics({k: v for k, v in metrics.items() if k != "threshold"})
    mlflow.lightgbm.log_model(lgb, artifact_path="model")
    print(metrics)

## 6. StratifiedKFold + multi-modèles + seuil métier

Les baselines des sections 4–5 (seuil 0.5, split unique) restent pour l’historique UI.

- **CV multi-algos** (`src/run_cv_baselines.py`) : StratifiedKFold 5 + régression logistique / random forest / LightGBM / XGBoost.
- **Optuna LightGBM** (`src/run_optuna_lgbm.py`) : 25 trials, objectif AUC en validation croisée ; puis seuil* sur valid.

```powershell
uv run python -m src.run_cv_baselines
uv run python -m src.run_optuna_lgbm --n-trials 25
```

Artefacts : `reports/cv_threshold_summary.csv`, `reports/optuna_lgbm_best.json`.


## 7. Comparer les runs CV dans l’UI

Protocole (même protocole = runs `*_cv5_seuil_opt_fe_v1`) :

1. UI : http://127.0.0.1:5000 → expérience `credit-scoring-fe_v1`
2. Vider les filtres de recherche
3. Cocher uniquement les runs CV5 terminées :
   - `logreg_cv5_seuil_opt_fe_v1`
   - `lightgbm_cv5_seuil_opt_fe_v1`
   - `xgboost_cv5_seuil_opt_fe_v1`
   - `random_forest_cv5_seuil_opt_fe_v1`
4. **Compare** → tableau **Metrics**
5. Métriques : `cv_auc_mean`, `valid_auc`, `valid_business_cost_at_0.5` (et `_at_opt`)

Ne pas mélanger dans ce Compare les runs `*_baseline_*` (pas de CV) ni `lightgbm_optuna_fe_v1` (autre question : gain d’hyperparamètres).


In [ ]:
# Tableau local = même comparaison que dans l'UI (runs CV5)
summary = pd.read_csv(ROOT / "reports" / "cv_threshold_summary.csv")
summary = summary.sort_values("valid_cost_0.5")
display_cols = [
    "model",
    "cv_auc_mean",
    "cv_auc_std",
    "valid_auc",
    "valid_cost_0.5",
    "valid_cost_at_oof_thr",
    "threshold_opt_oof",
]
summary[display_cols].round(4)

### Résultats CV5 (vérifiés — runs `*_cv5_seuil_opt_fe_v1`)

| Modèle | CV AUC mean | Valid AUC | Valid coût@0.5 (10:1) | Rang |
|--------|-------------|-----------|------------------------|------|
| **LightGBM** | **0.779** | **0.784** | **30013** | **1** |
| XGBoost | 0.774 | 0.780 | 30421 | 2 |
| Logistic Regression | 0.768 | 0.770 | 31666 | 3 |
| Random Forest | 0.757 | 0.762 | 32324 | 4 |

Le classement AUC (CV et valid) et le coût métier@0.5 donnent le **même ordre**. LightGBM est retenu comme champion de la phase CV.

Suite (déjà exécutée) :
- Optuna LightGBM → valid AUC ≈ **0.786**, seuil* valid = **0.52**, coût* ≈ **29962** (`reports/optuna_lgbm_best.json`) ;
- SHAP global / local → `reports/shap/` (`CODE_GENDER` apparaît dans le top d’importance : variable sensible, point d’attention) ;
- Registry + serving : `models:/credit-scoring-lgbm@champion` ; le seuil* **0.52** s’applique côté application (`proba >= 0.52` → refus).
